[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gamaly/dpgs-analysis-2026/blob/main/01_registry_and_sample.ipynb)


In [1]:
# ======================================================================================================
#  SETUP (inserted by the build script). In Google Colab this downloads the repository so that the
#  `data/` folder sits next to the notebook; on your own computer it does nothing at all.
# ======================================================================================================
import os, sys

REPO_SLUG = "gamaly/dpgs-analysis-2026"     # <github user>/<repository>; set in scripts/99b_build_public_repo.py
BRANCH = "main"

IN_COLAB = "google.colab" in sys.modules      # Colab imports this module in every session; a local Jupyter does not

if IN_COLAB and not os.path.isdir("data"):
    # Colab starts with an empty working directory, so the data folder is missing. Fetch the whole repository
    # (a few tens of MB): first try a shallow `git clone`, and if that fails download the branch zip instead.
    import subprocess, urllib.request, zipfile
    repo_dir = REPO_SLUG.split("/")[-1]
    try:
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                        f"https://github.com/{REPO_SLUG}.git", repo_dir], check=True)
    except Exception as err:
        print("git clone failed (", err, ") -> downloading the zip instead")
        url = f"https://github.com/{REPO_SLUG}/archive/refs/heads/{BRANCH}.zip"
        urllib.request.urlretrieve(url, "repo.zip")
        with zipfile.ZipFile("repo.zip") as z:
            z.extractall(".")                                   # unpacks to <repository>-<branch>/
        repo_dir = f"{repo_dir}-{BRANCH}"
    os.chdir(repo_dir)                                         # from here on, "data/" and "extras/" resolve as they do locally
    print("Colab: working in", os.getcwd())
    # Colab already has pandas, numpy and matplotlib. Install only what is missing (nothing, normally).
    import importlib.util
    missing = [p for p in ("pandas", "numpy", "matplotlib") if importlib.util.find_spec(p) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
elif not IN_COLAB:
    # Local use: run Jupyter from the repository folder (the one containing this notebook and data/).
    assert os.path.isdir("data"), "Start Jupyter in the repository folder: 'data/' must sit next to this notebook."


# 01. The registry, the confidence tiers and the sample

**What this notebook does.** It starts from the Digital Public Goods (DPG) registry and shows, step by step, how
256 listed DPGs become the **166 DPGs** and **190 repositories** that every later number in the post rests on.
It also reproduces the status counts (how many of the 166 are active, slowing or dormant).

**Why it matters.** The post says things like "a quarter of the certified registry has nobody shipping". That is
only meaningful if we are clear about *which* DPGs are counted. A DPG is in the main sample only if we are sure
which GitHub repositories are really its own (rule: only high-confidence data feed the primary results). Uncertain cases are left out by design, not hidden.

**Inputs (all in `data/`, no internet needed).**
`registry_dpgs.csv`, `dpg_repo_map.csv`, `dpgs_without_code_repo.csv`, `all_repos.csv`, `dpg_confidence.csv`,
`dpg_status.csv`, `dpg_no_cohort_repo.csv`, `pr_level.csv`.
See `extras/DATA_DICTIONARY.md` for every column.

**Reading guide.** Code cells have comments saying what each line is for. The last cell prints every number from the post that this notebook reproduces, by claim id.

In [2]:
# --- setup -------------------------------------------------------------------------------------
import sys
from pathlib import Path
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

DATA = Path("data")        # every notebook reads ONLY this folder (no API calls, no token)
# Each "Claim Nxxx" cell below stores the number(s) it reproduces with record_result(...); the last cell of the
# notebook prints them all with print_results(). (Nothing is read from or written to disk.)
RESULTS = {}

def record_result(claim_id, description, values=(), ok=None, note=""):
    # values: the numbers the cell reproduced; ok: a yes/no test for claims that are not a number
    RESULTS[claim_id] = {"description": description, "values": [float(v) for v in values], "ok": ok, "note": note}

def print_results():
    # plain summary: claim id, what it is, the numbers (or the yes/no test)
    for cid, r in RESULTS.items():
        shown = ", ".join(f"{v:.6g}" for v in r["values"]) if r["values"] else ("yes" if r["ok"] else "no")
        print(f"{cid}  {r['description']}\n      -> {shown}" + (f"   ({r['note']})" if r["note"] else ""))
    print(f"\n{len(RESULTS)} claims reproduced in this notebook")

## 1. The registry
The registry was pulled from the DPG Alliance API on 29 October 2026. `registry_dpgs.csv` is a stripped copy:
id, name, stage and website. (The raw registry also holds contact names and e-mail addresses of the people who
registered each DPG. Those are personal data and are deliberately not in this release.)

In [3]:
reg = pd.read_csv(DATA / "registry_dpgs.csv")
print("DPGs in the registry:", len(reg))
print(reg.stage.value_counts().to_string())   # all 256 are at stage "DPG" (certified)

# The post's opening names five tools from the earlier post. Which of them are on the registry?
# Names differ slightly: OpenDataKit is listed as "ODK".
tools = {"KoboToolbox": ["KoboToolbox"], "OpenDataKit": ["ODK", "OpenDataKit"],
         "GeoNode": ["GeoNode"], "QGIS": ["QGIS"], "Moodle": ["Moodle"]}
names = set(reg.name.str.strip())
found = {t: any(a in names for a in aliases) for t, aliases in tools.items()}
print(found)
n_tools_on_registry = sum(found.values())
print("tools on the registry:", n_tools_on_registry)    # post says three: KoboToolbox, ODK, QGIS

DPGs in the registry: 256
stage
DPG    256
{'KoboToolbox': True, 'OpenDataKit': True, 'GeoNode': False, 'QGIS': True, 'Moodle': False}
tools on the registry: 3


## 2. From DPG to code repository
`dpg_repo_map.csv` is the cleaned map from DPG to the repositories it lists (script `01_clean_registry.py`).
The registry field is messy (several URLs in one cell, links to LICENSE files, organisation-only URLs), so each
URL was normalised to `owner/repo`. `host` says what kind of link it was:
* `github`      a specific repository,
* `github_org`  only a GitHub *organisation* is listed (the DPG gives no single repo),
* `gitlab`      GitLab (not collected; phase 2).
DPGs with no code repository at all (data sets, content, standards) are in `dpgs_without_code_repo.csv`.

In [4]:
rmap = pd.read_csv(DATA / "dpg_repo_map.csv")
nocode = pd.read_csv(DATA / "dpgs_without_code_repo.csv")

by_host = rmap.groupby("host").dpgid.nunique()          # DISTINCT DPGs per host type (a DPG can have both a repo and an org)
print("DPGs by link type:\n", by_host.to_string())
n_org_only_listed = by_host["github_org"]               # DPGs that list a GitHub organisation
n_no_code = nocode.dpgid.nunique()
print("\nDPGs listing a GitHub organisation:", n_org_only_listed)
print("DPGs with no code repository at all:", n_no_code)

DPGs by link type:
 host
github        173
github_org     21
gitlab          9

DPGs listing a GitHub organisation: 21
DPGs with no code repository at all: 54


## 3. Repository inventory and confidence tiers
`all_repos.csv` lists every repository we considered for a DPG and a **confidence tier**:

| tier | meaning |
|---|---|
| **high** | listed directly in the registry, still exists (renames followed), not a fork or mirror, belongs to one DPG |
| medium | an organisation repo picked by a rule (name match / dedicated organisation), not hand-checked |
| low | shared between DPGs, cannot be attributed cleanly |
| excluded | does not resolve, is a fork/mirror, or does not match |

A **DPG** is `high` only if every repo attributed to it is high and no listed repo was lost.
The 27,746 organisation repos that were rejected by the name rule (almost all OpenMandriva package repos) are not
in the release; `all_repos_not_matched_summary.csv` counts them per DPG.

In [5]:
ar = pd.read_csv(DATA / "all_repos.csv")
conf = pd.read_csv(DATA / "dpg_confidence.csv")

print("Repo tiers (rows in the released inventory):")
print(ar.confidence.value_counts().to_string())
print("\nWhy repos are excluded:")
print(ar[ar.confidence == "excluded"].confidence_reason.value_counts().to_string())
print("\nDPG tiers:")
print(conf.dpg_confidence.value_counts().to_string())

# The main sample: DPGs whose tier is "high".
high_dpgs = set(conf.loc[conf.dpg_confidence == "high", "dpgid"])
n_high_dpgs = len(high_dpgs)

# Repos: high-confidence repos that belong to a high-confidence DPG, counted once per repository name.
# (198 repos are "high" in all; 8 of them belong to DPGs that are not high overall, e.g. because another repo of
# that DPG was lost. They are not in the sample.)
hi_repos = ar[(ar.confidence == "high")]
n_high_repos_all = hi_repos.full_name.nunique()
n_sample_repos = hi_repos[hi_repos.dpgid.isin(high_dpgs)].full_name.nunique()
print(f"\nhigh DPGs: {n_high_dpgs}; high repos in all: {n_high_repos_all}; repos in the 166 DPGs: {n_sample_repos}")

Repo tiers (rows in the released inventory):
confidence
medium      699
excluded    271
high        198
low           2

Why repos are excluded:
confidence_reason
fork_or_mirror               207
org_repo_stale_since_2020     33
org_repo_nonproduct_name      26
unresolved                     3
manual_exclude                 2

DPG tiers:
dpg_confidence
high              166
medium             23
no_usable_repo     11
low                 2

high DPGs: 166; high repos in all: 198; repos in the 166 DPGs: 190


## 4. Which part of the registry is in the sample?
The sample rule says to report what the filters remove. The table shows, for the DPGs that are *not* in the sample,
why. (Org-only, multi-repo and moved projects are the ones most likely to be lost, so the sample leans toward
single-repo projects. This is a limit of the study and is stated in the post.)

In [6]:
reg_ids = set(reg.dpgid)
with_code = set(rmap.dpgid)
summary = pd.Series({
    "registry DPGs": len(reg_ids),
    "no code repository at all": n_no_code,
    "have at least one listed repo/org": len(with_code),
    "...of which in the main (high) sample": len(with_code & high_dpgs),
})
print(summary.to_string())
print("\nDPG tiers among DPGs that have code (GitHub or GitLab):")
print(conf[conf.dpgid.isin(with_code)].dpg_confidence.value_counts().to_string())

registry DPGs                            256
no code repository at all                 54
have at least one listed repo/org        202
...of which in the main (high) sample    166

DPG tiers among DPGs that have code (GitHub or GitLab):
dpg_confidence
high              166
medium             23
no_usable_repo     11
low                 2


## 5. Status today: active, slowing, dormant
For each repo, an *event* is a merged pull request by a human, an AI agent or an unclassified account, or a
(non-prerelease) release. Counting over the 12 months to 29 Oct 2026:
* **dormant**: no event in the last 12 months;
* **slowing**: fewer than half as many events as in the 12 months before, and at least 6 then;
* **active**: everything else.
A DPG takes the **best** status of its repos (one active repo makes the DPG active).
Bots that only merge dependency updates do not count as activity (`bot_only` flags such cases).
`dpg_status.csv` has two repo sets; the post uses `repo_set == "high"`.

In [7]:
st = pd.read_csv(DATA / "dpg_status.csv")
s = st[(st.repo_set == "high") & st.dpgid.isin(high_dpgs)]       # one row per DPG in the sample
counts = s.status.value_counts()
print(counts.to_string(), "\ntotal:", len(s))
share_dormant = counts["dormant"] / len(s) * 100
print(f"\nshare dormant: {share_dormant:.1f}%")

# The memo's earlier 109/17/44 came from all 170 rows of repo_set == 'high', including 4 DPGs that are not in the
# 166. The post uses the 166-DPG filter, as above.
print("\nunfiltered 'high' rows:", st[st.repo_set == "high"].status.value_counts().to_dict())

status
active     106
dormant     43
slowing     17 
total: 166

share dormant: 25.9%

unfiltered 'high' rows: {'active': 109, 'dormant': 44, 'slowing': 17}


## 6. DPGs whose code base is too new to measure
The activity panel (notebook 02) follows a *fixed cohort* of repos: created before 1 Jan 2024, or with at least 50
commits before then (a repo that moved to GitHub keeps its history). This stops brand-new repos from inflating
output. DPGs with no cohort repo get **no panel rows** (never zeros) and are listed in `dpg_no_cohort_repo.csv`.

In [8]:
nc = pd.read_csv(DATA / "dpg_no_cohort_repo.csv")
n_nc_high = nc.dpgid.isin(high_dpgs).sum()
print("DPGs with no cohort repo:", len(nc), "| of them in the 166:", n_nc_high)

dq = pd.read_csv(DATA / "dpg_quarterly.csv")
in_panel = set(dq.dpgid) & high_dpgs
print("high-confidence DPGs with panel rows:", len(in_panel), "(", n_high_dpgs, "-", n_nc_high, "=", n_high_dpgs - n_nc_high, ")")

DPGs with no cohort repo: 23 | of them in the 166: 22
high-confidence DPGs with panel rows: 144 ( 166 - 22 = 144 )


## 7. Window of the data
History is collected from 1 January 2020 (the data window starts on 1 January 2020) to the day of collection, 29 October 2026. The last PR date in the
per-PR file shows the end of the window.

In [9]:
pr = pd.read_csv(DATA / "pr_level.csv", usecols=["merged_date", "created_date"])
last_date = max(pr.merged_date.max(), pr.created_date.max())
print("first quarter in panel:", dq.quarter.min(), "| latest PR date in per-PR file:", last_date)
WINDOW_START = "2020-01-01"      # the data window starts on 1 January 2020
y0, m0, d0 = [int(x) for x in WINDOW_START.split("-")]
y1, m1, d1 = [int(x) for x in last_date.split("-")]

first quarter in panel: 2020Q1 | latest PR date in per-PR file: 2026-10-29


## 8. Claims from the post covered here
One labelled cell per claim id (the ids used in the post's footnotes).

### Claim N001: Three of the five tools from the first post are on the registry

In [10]:
record_result('N001','tools of the first post on the registry (KoboToolbox, ODK, QGIS)',[n_tools_on_registry]); print('tools on registry:',n_tools_on_registry)

tools on registry: 3


### Claim N013: Twenty-one DPGs list only a GitHub organisation

In [11]:
record_result('N013','DPGs listing a GitHub organisation (host == github_org)',[n_org_only_listed]); print(n_org_only_listed)

21


### Claim N014: Fifty-four DPGs have no code repository

In [12]:
record_result('N014','DPGs with no code repository',[n_no_code]); print(n_no_code)

54


### Claim N017: 166 certified DPGs with confidently matched repositories

In [13]:
record_result('N017','high-confidence DPGs',[n_high_dpgs]); print(n_high_dpgs)

166


### Claim N018: 190 repositories in those 166 DPGs

In [14]:
record_result('N018','repos in the 166 DPGs (190); 198 high repos in all',[n_sample_repos,n_high_repos_all]); print(n_sample_repos,'(198 high repos in all:',n_high_repos_all,')')

190 (198 high repos in all: 198 )


### Claim N019: Window: 1 Jan 2020 to 29 Oct 2026; registry of 256 DPGs

In [15]:
record_result('N019','window 2020-01-01..end date; registry size',[y0,m0,d0,y1,m1,d1,len(reg)]); print(WINDOW_START,last_date,len(reg))

2020-01-01 2026-10-29 256


### Claim N046: 106 active, 17 slowing, 43 dormant

In [16]:
record_result('N046','status counts among the 166',[counts['active'],counts['slowing'],counts['dormant']]); print(counts.to_dict())

{'active': 106, 'dormant': 43, 'slowing': 17}


### Claim N047: A quarter of the registry has nobody shipping (43 of 166)

In [17]:
record_result('N047','share dormant (percent)',[share_dormant]); print(round(share_dormant,1))

25.9


### Claim N089: Twenty-two of the 166 have code bases that began in 2024 or later

In [18]:
record_result('N089','DPGs in the 166 with no cohort repo',[n_nc_high]); print(n_nc_high)

22


### Claim N092: in 13 of the 43 dormant DPGs someone still committed directly, one of them 810 times
"Dormant" = no merged PR or release in 12 months. This counts *commits by humans* in those 12 months
(`human_commits_12m` in `repo_status.csv`, summed over the DPG's high-confidence repos). Direct commits are not
counted as activity by the status rule, so a dormant DPG can still show commits.

In [19]:
rs = pd.read_csv(DATA / "repo_status.csv")
rs["k"] = rs.full_name.str.lower()
hr_repos = ar[(ar.confidence == "high") & ar.dpgid.isin(high_dpgs)].drop_duplicates(["dpgid", "full_name"]).assign(k=lambda x: x.full_name.str.lower())
hc = hr_repos.merge(rs[["k", "human_commits_12m"]], on="k", how="left").groupby("dpgid").human_commits_12m.sum()
dormant_ids = list(s.loc[s.status == "dormant", "dpgid"])
dc = hc.reindex(dormant_ids).fillna(0)
n_commit = int((dc > 0).sum())
print(f"dormant DPGs: {len(dormant_ids)} | with human commits in the last 12 months: {n_commit} | most commits: {int(dc.max())}")
record_result("N092", "dormant DPGs with human commits; of dormant; max commits", [n_commit, len(dormant_ids), dc.max()])

dormant DPGs: 43 | with human commits in the last 12 months: 13 | most commits: 810


## 8b. More claims: how the dormancy count depends on the window, and three registry descriptions

### Claim N105: the dormancy count depends on the window (6, 12, 18 and 24 months)
The headline status (notebook section 5) uses a 12-month window. Here the same rule is re-run with other window lengths on
**all 170 DPGs that have a high-confidence repository** (the 166 of the sample plus 4 whose other repositories were lost).
`repo_events.csv` holds dated event counts per repository (merged PRs by humans, AI agents or unknown accounts; releases;
direct commits by humans). A repository is *dormant* when it has no event in the last `m` months, *slowing* when the last `m`
months have fewer than half the events of the `m` months before (and at least `6*m/12` of them), else *active*.
A DPG takes the best status of its repositories. A second run also counts **direct human commits** as activity.

In [20]:
# --- dated events per repo (built by the build script from the raw download; see data dictionary) -----------------------
ev = pd.read_csv(DATA / "repo_events.csv")                  # columns: full_name, date, kind (merge | release | commit), n
rst = pd.read_csv(DATA / "repo_status.csv")                 # one row per collected repository (has pushed_at)
REF = pd.Timestamp("2026-10-29")                            # collection date: the end of every window

# An event is stored by DAY. Treat it as happening at the END of that day. The windows are (start, end]; with end-of-day
# stamps the comparison gives the same answer as the original timestamps, except for an event stamped exactly 00:00:00.
ev["t"] = pd.to_datetime(ev["date"]) + pd.Timedelta(hours=23, minutes=59, seconds=59)
ev["fk"] = ev.full_name.str.lower()
by_repo = {k: g for k, g in ev.groupby("fk")}                # events grouped per repository (lower-case name)

# Repositories: every high-confidence row of the inventory, one row per (DPG, repository), that was actually collected.
collected = set(rst.full_name.str.lower())
hi_all = ar[ar.confidence == "high"].drop_duplicates(["dpgid", "full_name"]).copy()
hi_all["fk"] = hi_all.full_name.str.lower()
hi_all = hi_all[hi_all.fk.isin(collected)]

RANK = {"active": 0, "slowing": 1, "dormant": 2}            # the DPG takes the BEST (lowest-rank) status of its repos

def repo_status(times, counts, months):
    # times/counts: event timestamps and how many events happened at each
    r0 = REF - pd.DateOffset(months=months)                 # start of the recent window
    p0 = REF - pd.DateOffset(months=2 * months)             # start of the window before it
    recent = counts[(times > r0) & (times <= REF)].sum()
    prior = counts[(times > p0) & (times <= r0)].sum()
    if recent == 0:
        return "dormant"
    if recent < 0.5 * prior and prior >= max(1, round(6 * months / 12)):
        return "slowing"
    return "active"

def dpg_status_for(months, kinds):
    out = {}
    for dpg, g in hi_all.groupby("dpgid"):
        sts = []
        for fk in g.fk:
            e = by_repo.get(fk)
            if e is None:                                   # collected repo with no event in the stored range: dormant
                sts.append("dormant"); continue
            e = e[e.kind.isin(kinds)]
            sts.append(repo_status(e.t.to_numpy(), e.n.to_numpy(), months))
        out[dpg] = min(sts, key=RANK.get)
    return pd.Series(out)

MERGE_REL = ["merge", "release"]
res = {m: dpg_status_for(m, MERGE_REL) for m in (6, 12, 18, 24)}
n_dpgs_hi = len(res[12])
dormant_by_window = {m: int((s == "dormant").sum()) for m, s in res.items()}
print("DPGs with a high-confidence repository:", n_dpgs_hi)
print("dormant by window (months):", dormant_by_window, "| slowing at 12:", int((res[12] == "slowing").sum()))
print("share dormant %:", {m: round(v / n_dpgs_hi * 100, 1) for m, v in dormant_by_window.items()})

# With direct human commits counted as activity (12-month window)
res_c = dpg_status_for(12, MERGE_REL + ["commit"])
n_dormant_commits = int((res_c == "dormant").sum())

# Of the 44 dormant DPGs (12 months, no commits), how many have a repository pushed to since 1 January 2026?
dormant12 = set(res[12][res[12] == "dormant"].index)
pushed = rst.assign(fk=rst.full_name.str.lower()).set_index("fk").pushed_at
recent_push = {dpg for dpg in dormant12
               if (pushed.reindex(hi_all.loc[hi_all.dpgid == dpg, "fk"]).dropna().astype(str) >= "2026-01-01").any()}
print("dormant at 12 months but with direct commits counted:", n_dormant_commits, "| still dormant DPGs with a repo pushed since 2026-01-01:", len(recent_push))
record_result("N105", "dormant DPGs at 6, 12, 18, 24 months (of 170); 12 months with direct commits; dormant DPGs with a repo pushed since 2026-01-01",
         [dormant_by_window[6], dormant_by_window[12], dormant_by_window[18], dormant_by_window[24], n_dpgs_hi,
          n_dormant_commits, len(recent_push)])

DPGs with a high-confidence repository: 170
dormant by window (months): {6: 51, 12: 44, 18: 32, 24: 29} | slowing at 12: 17
share dormant %: {6: 30.0, 12: 25.9, 18: 18.8, 24: 17.1}
dormant at 12 months but with direct commits counted: 31 | still dormant DPGs with a repo pushed since 2026-01-01: 17


### Claim N122: three examples of what the software on the list does
The registry's own one-sentence descriptions of three DPGs: Bahmni (clinic and hospital records), the Immunization Calculation
Engine (which vaccines a child is due) and FUSS (the computers in schools). The check is that each description contains the
words the post paraphrases.

In [21]:
# `registry_dpgs.csv` carries the public one-sentence description of each DPG (no contact details).
desc = reg.assign(low=reg.description.fillna("").str.lower()).set_index("name").low
needles = {"Bahmni": "clinics and hospitals",
           "Immunization Calculation Engine": "immunization evaluation and forecasting",
           "FUSS - Free Upgrade for a digitally Sustainable School": "schools"}
found_desc = {k: (k in desc.index and v in desc[k]) for k, v in needles.items()}
print(found_desc)
record_result("N122", "registry descriptions that support the three examples", [sum(found_desc.values())], ok=all(found_desc.values()))

{'Bahmni': True, 'Immunization Calculation Engine': True, 'FUSS - Free Upgrade for a digitally Sustainable School': True}


## 9. Summary of the numbers reproduced here

In [22]:
print_results()

N001  tools of the first post on the registry (KoboToolbox, ODK, QGIS)
      -> 3
N013  DPGs listing a GitHub organisation (host == github_org)
      -> 21
N014  DPGs with no code repository
      -> 54
N017  high-confidence DPGs
      -> 166
N018  repos in the 166 DPGs (190); 198 high repos in all
      -> 190, 198
N019  window 2020-01-01..end date; registry size
      -> 2020, 1, 1, 2026, 10, 29, 256
N046  status counts among the 166
      -> 106, 17, 43
N047  share dormant (percent)
      -> 25.9036
N089  DPGs in the 166 with no cohort repo
      -> 22
N092  dormant DPGs with human commits; of dormant; max commits
      -> 13, 43, 810
N105  dormant DPGs at 6, 12, 18, 24 months (of 170); 12 months with direct commits; dormant DPGs with a repo pushed since 2026-01-01
      -> 51, 44, 32, 29, 170, 31, 17
N122  registry descriptions that support the three examples
      -> 3

12 claims reproduced in this notebook
